# ML Experiments — Wheel Strategy Options Competition

**Goal:** Find the best ML approach to predict:
1. **Assignment Risk** — will this put be assigned? (classification)
2. **Optimal Close Day** — when to take profit (regression)
3. **Stock Selection** — which stocks give best risk-adjusted premium (ranking)

**Data:** 2 years of IBKR historical data (69 tickers, ~31K examples)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    mean_absolute_error, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load base dataset
df = pd.read_csv('../data/processed/ml_dataset_otm5_dte30.csv')
print(f"Dataset: {len(df):,} rows | {df.ticker.nunique()} tickers")
print(f"Date range: {df.date.min()} → {df.date.max()}")
print(f"Assignment rate: {df.was_assigned.mean()*100:.1f}%")
df.head()

## 1. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Assignment rate by ticker (top 20)
assign_by_ticker = df.groupby('ticker')['was_assigned'].mean().sort_values(ascending=False).head(20)
assign_by_ticker.plot.barh(ax=axes[0,0], color='salmon')
axes[0,0].set_title('Assignment Rate by Ticker (Top 20)')
axes[0,0].set_xlabel('Assignment Rate')

# 2. Premium yield distribution
df['premium_yield'].hist(bins=50, ax=axes[0,1], color='steelblue', alpha=0.7)
axes[0,1].axvline(df['premium_yield'].median(), color='red', linestyle='--', label=f"median={df['premium_yield'].median():.2f}")
axes[0,1].set_title('Premium Yield Distribution')
axes[0,1].legend()

# 3. IV Rank vs Assignment
df_clean = df.dropna(subset=['iv_rank'])
assigned = df_clean[df_clean.was_assigned==1]['iv_rank']
not_assigned = df_clean[df_clean.was_assigned==0]['iv_rank']
axes[0,2].hist(not_assigned, bins=30, alpha=0.6, label='OTM (win)', color='green')
axes[0,2].hist(assigned, bins=30, alpha=0.6, label='Assigned', color='red')
axes[0,2].set_title('IV Rank: Assigned vs OTM')
axes[0,2].legend()

# 4. RSI vs Assignment rate (binned)
df['rsi_bin'] = pd.cut(df['rsi'], bins=10)
rsi_assign = df.groupby('rsi_bin')['was_assigned'].mean()
rsi_assign.plot.bar(ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Assignment Rate by RSI Bin')
axes[1,0].tick_params(axis='x', rotation=45)

# 5. Trend score vs assignment
trend_assign = df.groupby('trend_score')['was_assigned'].mean()
trend_assign.plot.bar(ax=axes[1,1], color=['red','orange','yellowgreen','green'])
axes[1,1].set_title('Assignment Rate by Trend Score')
axes[1,1].set_ylabel('Assignment Rate')

# 6. PnL distribution
df['pnl'].hist(bins=50, ax=axes[1,2], color='steelblue', alpha=0.7)
axes[1,2].axvline(0, color='red', linestyle='--')
axes[1,2].set_title(f"PnL Distribution (mean=${df.pnl.mean():.3f})")

plt.tight_layout()
plt.savefig('../data/processed/eda_overview.png', dpi=150)
plt.show()
print(f"\nCorrelation with assignment:")
print(df[['premium_yield','iv_est','iv_rank','rv20','rsi','trend_score','return_20d','atr_pct','was_assigned']].corr()['was_assigned'].sort_values())

## 2. Advanced Feature Engineering

New features to test:
- **Lag features:** momentum over different windows (5d, 10d, 60d)
- **Interaction features:** IV × momentum, RSI × trend
- **Sector encoding:** one-hot for sector
- **Volatility regime:** high/low vol environment
- **Mean-reversion signals:** distance from MA, Bollinger bands
- **Volume features:** relative volume vs average

In [ ]:
# Load raw IBKR data for advanced feature engineering
raw = pd.read_csv('../data/raw/stock_history_ibkr.csv')
raw['date'] = raw['date'].astype(str)

def build_advanced_features(raw_df):
    """Build richer feature set from raw OHLCV data."""
    results = []
    
    for ticker, g in raw_df.groupby('ticker'):
        g = g.sort_values('date').copy()
        c = g['close']
        h = g['high']
        l = g['low']
        v = g['volume']
        
        # --- Trend features ---
        for w in [5, 10, 20, 50]:
            g[f'sma{w}'] = c.rolling(w).mean()
            g[f'return_{w}d'] = c.pct_change(w)
        
        g['above_sma20'] = (c > g['sma20']).astype(int)
        g['above_sma50'] = (c > g['sma50']).astype(int)
        g['sma20_above_sma50'] = (g['sma20'] > g['sma50']).astype(int)
        g['trend_score'] = g['above_sma20'] + g['above_sma50'] + g['sma20_above_sma50']
        
        # --- Volatility features ---
        log_ret = np.log(c / c.shift(1))
        for w in [10, 20, 60]:
            g[f'rv{w}'] = log_ret.rolling(w).std() * np.sqrt(252)
        
        g['iv_est'] = g['rv20'] * 1.15
        
        # Volatility ratio (short vs long)
        g['vol_ratio'] = g['rv10'] / g['rv60'].replace(0, np.nan)
        
        # IV rank
        g['iv_rank'] = g['rv20'].rolling(252, min_periods=50).apply(
            lambda x: (x.iloc[-1]-x.min())/(x.max()-x.min())*100 
            if x.max()!=x.min() else 50, raw=False)
        
        # --- RSI ---
        delta = c.diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss.replace(0, np.nan)
        g['rsi'] = 100 - (100 / (1 + rs))
        
        # --- ATR ---
        tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
        g['atr_pct'] = tr.rolling(14).mean() / c * 100
        
        # --- Bollinger Bands ---
        g['bb_upper'] = g['sma20'] + 2 * c.rolling(20).std()
        g['bb_lower'] = g['sma20'] - 2 * c.rolling(20).std()
        g['bb_position'] = (c - g['bb_lower']) / (g['bb_upper'] - g['bb_lower'])  # 0-1
        
        # --- Volume features ---
        g['vol_sma20'] = v.rolling(20).mean()
        g['relative_volume'] = v / g['vol_sma20'].replace(0, np.nan)
        
        # --- Mean-reversion ---
        g['dist_sma20_pct'] = (c - g['sma20']) / g['sma20'] * 100
        g['dist_sma50_pct'] = (c - g['sma50']) / g['sma50'] * 100
        
        # --- 52-week metrics ---
        g['dist_52w_high'] = (c.rolling(252, min_periods=50).max() - c) / c * 100
        g['dist_52w_low'] = (c - c.rolling(252, min_periods=50).min()) / c * 100
        
        # --- Consecutive up/down days ---
        g['up_day'] = (c > c.shift(1)).astype(int)
        g['consec_up'] = g['up_day'].groupby((g['up_day'] != g['up_day'].shift()).cumsum()).cumsum()
        g['consec_down'] = (1 - g['up_day']).groupby(((1-g['up_day']) != (1-g['up_day']).shift()).cumsum()).cumsum()
        
        # --- Interaction features ---
        g['iv_x_momentum'] = g['iv_est'] * g['return_20d']
        g['rsi_x_trend'] = g['rsi'] * g['trend_score'] / 3
        g['vol_x_atr'] = g['rv20'] * g['atr_pct']
        
        results.append(g)
    
    return pd.concat(results).reset_index(drop=True)

raw_feat = build_advanced_features(raw)
print(f"Advanced features: {raw_feat.shape}")
print(f"New columns: {[c for c in raw_feat.columns if c not in raw.columns]}")